In [1]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
import time

In [2]:
# Read the dataset
df = pd.read_csv(r'D:\Downloads\ml_features_and_labels.csv')

# Use the provided split column to create train/test sets and exclude metadata
# columns that would leak information about the label.
feature_cols = [c for c in df.columns if c not in ['label', 'ID', 'split', 'taxonomy']]

X_train = df.loc[df['split'] == 'train', feature_cols]
y_train = df.loc[df['split'] == 'train', 'label']

X_test = df.loc[df['split'] == 'test', feature_cols]
y_test = df.loc[df['split'] == 'test', 'label']

In [3]:
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight=(neg_count / pos_count)
scale_pos_weight

6.986027944111776

In [8]:
# Method 2: Using RandomizedSearchCV (Scikit-Learn Native)
from sklearn.model_selection import RandomizedSearchCV
import lightgbm as lgb
from scipy.stats import randint, uniform

# 1. Define model
lgbm_model = lgb.LGBMClassifier(scale_pos_weight=scale_pos_weight, random_state=42)

# 2. Define parameter grid
param_distributions = {
    'n_estimators': randint(500, 2000),
    'max_depth': randint(5, 12),
    'learning_rate': uniform(0.05, 0.29),
    'num_leaves': randint(20, 150),
    'subsample': uniform(0.5, 0.5),
    'colsample_bytree': uniform(0.5, 0.5)
}

# 3. Setup RandomizedSearch
random_search = RandomizedSearchCV(
    estimator=lgbm_model,
    param_distributions=param_distributions,
    n_iter=40,          # Number of parameter settings to sample
    scoring='f1',       # Metric to optimize
    cv=9,               # Cross-validation folds
    verbose=1,
    random_state=42,
    n_jobs=-1           # Use all CPU cores
)

# 4. Execute search
random_search.fit(X_train, y_train)

print("Best parameters found: ", random_search.best_params_)
print("Best score: ", random_search.best_score_)

# The best model is automatically saved
best_model = random_search.best_estimator_

Fitting 9 folds for each of 40 candidates, totalling 360 fits
[LightGBM] [Info] Number of positive: 3006, number of negative: 21000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000681 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 697
[LightGBM] [Info] Number of data points in the train set: 24006, number of used features: 25
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.125219 -> initscore=-1.943912
[LightGBM] [Info] Start training from score -1.943912
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightG

In [10]:
from sklearn.metrics import classification_report, confusion_matrix

# 1. Make class predictions on unseen test data
y_pred = best_model.predict(X_test)

# 2. Evaluate the model's performance
print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# 3. View Feature Importances (works for XGBoost, LightGBM, CatBoost)
importances = best_model.feature_importances_
print("Feature Importances:", importances)

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98      7000
           1       0.90      0.83      0.87      1002

    accuracy                           0.97      8002
   macro avg       0.94      0.91      0.92      8002
weighted avg       0.97      0.97      0.97      8002

Confusion Matrix:
[[6909   91]
 [ 167  835]]
Feature Importances: [ 1311  9053 11122 21106 21325    76    56   589     2     0     0     0
    61     5     0     0     1     3     0     0     0     0     0     0
     0     0     0     0     0     0     0     0]
